In [1]:
import yaml
from app.datasets.loader import load_multiple_test_cases, load_test_cases
from app.datasets.validator import validate_dataset_schema
from app.client.rag_client import RAGClient
from app.tests.run_tests import run_tests
from app.tests.nodes.reformulate import send_reformulate_requests, save_reformualte_responses, reformulate_tests
from app.utils.db import save_results_on_cosmos

In [4]:
file_list = [
  # './app/data/raw/tramites.xlsx',
  # './app/data/raw/accesibilidad.xlsx',
  # './app/data/raw/descubrir.xlsx', 
  # './app/data/raw/solicitudes.xlsx',
  # './app/data/raw/organigrama.xlsx'
]

test_config = {
  'GENERAL_TESTS': False,
  'TIMINGS': {'test': False, 'report': False},
  'TOKENS': {'test': False, 'report': False},
  'FOUNDRYS': {'test': False, 'report': False},
  'TRIAGE': {'test': False, 'report': False},
  'ROUTER': {'test': False, 'report': False},
  'GROUNDING': {'test': False, 'report': False},
  'SAVE_RESULTS': False,
  'PATH': './app/data/processed/reports/report',
  
  'REFORMULATE': {'test': True, 'report': False}
}   

if file_list: 
  df = load_multiple_test_cases(file_list)
  df = validate_dataset_schema(df)

with open('./app/config/config.yaml', 'r') as file:
  config_data = yaml.load(file, Loader= yaml.FullLoader) 
  
client = RAGClient(config_data)
test_timestamps = {}

In [5]:
# TEST GENERALES
if test_config.get('GENERAL_TESTS', False):
  responses = client.query_batch(df['user_input'],df['reference'])
  save_responses_in_json, response_file_path = client.save_api_responses(responses)
  test_timestamps['general_tests'] = response_file_path

# SOLO REFORMULATE
if test_config.get('REFORMULATE', False).get('test', False):
  reformulate_dataset = load_test_cases('./app/data/raw/reformulate.xlsx')
  reformulate_results = send_reformulate_requests(config= config_data, dataset=reformulate_dataset)
  generate_reformulate_json, reformulate_timestamp = save_reformualte_responses(reformulate_results)
  test_timestamps['reformulate_test'] = reformulate_timestamp
  reformulate_results = reformulate_tests(reformulate_results)

In [6]:
print(reformulate_results)

4.6


In [4]:
import json

response_file_path = './app/data/processed/outcome_20260406-153646.json'
with open(response_file_path, 'r', encoding='UTF-8') as f:
  responses = json.load(f)
test_timestamps['general_tests'] = 'outcome_20260406-153646.json'

In [5]:
if test_config.get('GENERAL_TESTS'):
  results, reports = run_tests(
    config = test_config, 
    data = responses, 
    df = df, 
    timestamp = test_timestamps
  )

In [ ]:
print(results)

{'timestamp': '20260406-153646', 'nodes': {'triage': {'positives': 156, 'total': 163, 'result': 95.71}, 'router': {'positives': 152, 'total': 156, 'result': 97.44}, 'grounding': {'positives': 131, 'total': 156, 'result': 83.97}}}


In [ ]:
save_results_on_cosmos(results)